# SparkSession Entry point: SparkSession is the main entry point for using the PySpark SQL and DataFrame API.


#                            SparkSession      connection to the Spark cluster.

It allows you to:

Create DataFrames

Run SQL queries

Read/write data (CSV, Parquet, JSON, etc.)

# .builder SparkSession.builder returns a builder object that helps configure and create the session.

#You chain various configuration options (like app name, master URL, configs, etc.) before calling .getOrCreate().

# .appName("Read Parquet Employee Data") Sets a human-readable name for your Spark application.

You’ll see this name in:

Spark UI (on port 4040)

Cluster monitoring tools (YARN, Dataproc, etc.)

# .master("local[*]")   "local[*]" tells Spark to: “Run this job locally using as many worker threads as the number of logical CPU cores.”

Defines where your Spark job will run.

Master Setting	                    Meaning
"local"	                            Run on your local machine, single thread
"local[4]"	                        Run locally with 4 threads
"local[*]"	                        Run locally using all available CPU cores
"spark://host:port"	                Connect to a standalone Spark cluster
"yarn"	                            Run on Hadoop YARN cluster
"k8s://..."	                        Run on Kubernetes cluster


# .getOrCreate() This creates a new SparkSession if one doesn’t exist, or returns the existing one if it does.

This avoids creating duplicate sessions (which can cause resource conflicts).

# spark The variable spark now holds the SparkSession object, so you can use it to run Spark commands:

df = spark.read.parquet("employee_data.parquet")
df.show()

In [ ]:
# Creating DataFrames

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DF Example").getOrCreate()

data = [("Alice", 25), ("Bob", 30), ("Cathy", 28)]
columns = ["Name", "Age"]

df = spark.createDataFrame(data, columns)
df.show()

display(df)


+-----+---+
| Name|Age|
+-----+---+
|Alice| 25|
|  Bob| 30|
|Cathy| 28|
+-----+---+



DataFrame[Name: string, Age: bigint]

In [ ]:
data = [("Alice", 25, 5000), ("Bob", 30, 7000)]
columns = ["name", "age", "salary"]
df = spark.createDataFrame(data, columns)

# select

from pyspark.sql.functions import col, upper

df.select(
    upper(col("name")).alias("NAME"),
    (col("age") + 5).alias("AgeIn5Years"),
    (col("salary") * 1.1).alias("IncreasedSalary")
).show()


# selectExpr
df.selectExpr(
    "upper(name) as NAME",
    "age + 5 as AgeIn5Years",
    "salary * 1.1 as IncreasedSalary"
).show()


+-----+-----------+-----------------+
| NAME|AgeIn5Years|  IncreasedSalary|
+-----+-----------+-----------------+
|ALICE|         30|           5500.0|
|  BOB|         35|7700.000000000001|
+-----+-----------+-----------------+

+-----+-----------+---------------+
| NAME|AgeIn5Years|IncreasedSalary|
+-----+-----------+---------------+
|ALICE|         30|         5500.0|
|  BOB|         35|         7700.0|
+-----+-----------+---------------+



In [ ]:
# filter and where

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

#spark = SparkSession.builder.appName("FilterExample").getOrCreate()

data = [("Alice", 25), ("Bob", 30), ("Cathy", 22), ("David", 35)]

columns = ["name", "age"]

df = spark.createDataFrame(data, columns)


# filter

df.filter(col("age") > 25).show()


# Using where():

df.where(col("age") > 25).show()


# output for both
#+-----+---+
#| name|age|
#+-----+---+
#|  Bob| 30|
#|David| 35|


+-----+---+
| name|age|
+-----+---+
|  Bob| 30|
|David| 35|
+-----+---+

+-----+---+
| name|age|
+-----+---+
|  Bob| 30|
|David| 35|
+-----+---+



In [ ]:
# adding new col / modify existing col using withcolumn

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("withColumnExample").getOrCreate()

data = [("Alice", 25), ("Bob", 30)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)

# Add new column "age_plus_5"
df2 = df.withColumn("age_plus_5", col("age") + 5)
df2.show()

# modify existing column

df3 = df.withColumn("age", col("age") + 1)
df3.show()

+-----+---+----------+
| name|age|age_plus_5|
+-----+---+----------+
|Alice| 25|        30|
|  Bob| 30|        35|
+-----+---+----------+

+-----+---+
| name|age|
+-----+---+
|Alice| 26|
|  Bob| 31|
+-----+---+



# lit() is a function that allows you to add a constant (literal) value as a new column or in expressions.



In [ ]:
from pyspark.sql.functions import lit

df5 = df.withColumn("country", lit(None))
df5.show()


+-----+---+-------+
| name|age|country|
+-----+---+-------+
|Alice| 25|   NULL|
|  Bob| 30|   NULL|
+-----+---+-------+



# withColumnRenamed() is used to rename an existing column in a Spark DataFrame.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RenameColumn").getOrCreate()

data = [("Alice", 25, "India"), ("Bob", 30, "USA")]
columns = ["name", "age", "country"]

df = spark.createDataFrame(data, columns)
df.show()

df5 = df.withColumnRenamed("country", "Ctry")
df5.show()



+-----+---+-------+
| name|age|country|
+-----+---+-------+
|Alice| 25|  India|
|  Bob| 30|    USA|
+-----+---+-------+

+-----+---+-----+
| name|age| Ctry|
+-----+---+-----+
|Alice| 25|India|
|  Bob| 30|  USA|
+-----+---+-----+



# Both orderBy() and sort() are used to sort the rows of a DataFrame based on one or more columns (ascending or descending order).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = SparkSession.builder.appName("OrderBySortExample").getOrCreate()

data = [("Alice", 25, 3000),
        ("Bob", 30, 4000),
        ("Cathy", 22, 3500),
        ("David", 35, 2000)]
columns = ["name", "age", "salary"]

df = spark.createDataFrame(data, columns)
df.show()


+-----+---+------+
| name|age|salary|
+-----+---+------+
|Alice| 25|  3000|
|  Bob| 30|  4000|
|Cathy| 22|  3500|
|David| 35|  2000|
+-----+---+------+



In [2]:
df.orderBy("age").show()

+-----+---+------+
| name|age|salary|
+-----+---+------+
|Cathy| 22|  3500|
|Alice| 25|  3000|
|  Bob| 30|  4000|
|David| 35|  2000|
+-----+---+------+



In [3]:
df.orderBy(col("age").desc()).show()

+-----+---+------+
| name|age|salary|
+-----+---+------+
|David| 35|  2000|
|  Bob| 30|  4000|
|Alice| 25|  3000|
|Cathy| 22|  3500|
+-----+---+------+



In [4]:
df.sort("age").show()

+-----+---+------+
| name|age|salary|
+-----+---+------+
|Cathy| 22|  3500|
|Alice| 25|  3000|
|  Bob| 30|  4000|
|David| 35|  2000|
+-----+---+------+



In [5]:
df.sort(col("age").desc()).show()

+-----+---+------+
| name|age|salary|
+-----+---+------+
|David| 35|  2000|
|  Bob| 30|  4000|
|Alice| 25|  3000|
|Cathy| 22|  3500|
+-----+---+------+



# distinct() removes duplicate rows from a DataFrame and returns only unique records.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DistinctExample").getOrCreate()

data = [
    ("Alice", 25, "F"),
    ("Bob", 30, "M"),
    ("Alice", 25, "F"),
    ("David", 35, "M")
]
columns = ["name", "age", "gender"]

df = spark.createDataFrame(data, columns)
df.show()


In [ ]:
df.distinct().show()

```

Feature	            distinct()	                    dropDuplicates()
Deduplication       Scope	All columns	            Specific columns
Example	            df.distinct()	                df.dropDuplicates(["name"])
Use Case	        Remove entire row duplicates	Remove based on selected fields
Performance	        Slightly faster (less metadata)	Slightly slower (column tracking)

```



# Aggregation and group by

In [6]:

from pyspark.sql.functions import col, lit

df_transformed = (
    df
    .filter(col("Age") > 25)
    .withColumn("Country", lit("India"))
    .withColumnRenamed("Name", "FullName")
    .select("FullName", "Age", "Country")
)
df_transformed.show()


from pyspark.sql.functions import avg, max, min, count

df_transformed.groupBy("Country").agg(
    count("*").alias("Total"),
    avg("Age").alias("AvgAge"),
    max("Age").alias("MaxAge")
).show()


+--------+---+-------+
|FullName|Age|Country|
+--------+---+-------+
|     Bob| 30|  India|
|   David| 35|  India|
+--------+---+-------+

+-------+-----+------+------+
|Country|Total|AvgAge|MaxAge|
+-------+-----+------+------+
|  India|    2|  32.5|    35|
+-------+-----+------+------+



In [ ]:
# SQL Integration

df.createOrReplaceTempView("people")
spark.sql("SELECT Name, Age FROM people WHERE Age > 25").show()


+----+---+
|Name|Age|
+----+---+
| Bob| 30|
+----+---+



# Joins A join combines rows from two DataFrames based on a related column (a key).

# PySpark provides SQL-style joins through the DataFrame API.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.master("local[*]").appName("DataFrame Joins").getOrCreate()

customers = spark.createDataFrame([
    (1, "Alice", "Bangalore"),
    (2, "Bob", "Delhi"),
    (3, "Charlie", "Chennai")
], ["cust_id", "cust_name", "city"])

orders = spark.createDataFrame([
    (101, 1, "2025-11-01", 2500),
    (102, 2, "2025-11-02", 1800),
    (103, 4, "2025-11-03", 2200)
], ["order_id", "cust_id", "order_date", "amount"])


# Inner Join   returns only matching rows from both DataFrames.  

In [ ]:
inner_df = customers.join(orders, on="cust_id", how="inner") # Only customers with matching orders appear.
inner_df.show()


+-------+---------+---------+--------+----------+------+
|cust_id|cust_name|     city|order_id|order_date|amount|
+-------+---------+---------+--------+----------+------+
|      1|    Alice|Bangalore|     101|2025-11-01|  2500|
|      2|      Bob|    Delhi|     102|2025-11-02|  1800|
+-------+---------+---------+--------+----------+------+



# Left Join (Left Outer)  Keeps all rows from the left DataFrame, adds matching data from the right.

In [ ]:
left_df = customers.join(orders, on="cust_id", how="left") # Shows customers without orders (null values).
left_df.show()


+-------+---------+---------+--------+----------+------+
|cust_id|cust_name|     city|order_id|order_date|amount|
+-------+---------+---------+--------+----------+------+
|      1|    Alice|Bangalore|     101|2025-11-01|  2500|
|      3|  Charlie|  Chennai|    NULL|      NULL|  NULL|
|      2|      Bob|    Delhi|     102|2025-11-02|  1800|
+-------+---------+---------+--------+----------+------+



# Right Join (Right Outer) Keeps all rows from the right DataFrame, adds matching data from the left.


In [ ]:
right_df = customers.join(orders, on="cust_id", how="right") # Shows orders without matching customers.
right_df.show()


+-------+---------+---------+--------+----------+------+
|cust_id|cust_name|     city|order_id|order_date|amount|
+-------+---------+---------+--------+----------+------+
|      1|    Alice|Bangalore|     101|2025-11-01|  2500|
|      2|      Bob|    Delhi|     102|2025-11-02|  1800|
|      4|     NULL|     NULL|     103|2025-11-03|  2200|
+-------+---------+---------+--------+----------+------+



# Full Outer Join Keeps all rows from both DataFrames (fills missing values with null).

In [ ]:
full_df = customers.join(orders, on="cust_id", how="outer") # Shows all customers and all orders, even if unmatched.
full_df.show()


+-------+---------+---------+--------+----------+------+
|cust_id|cust_name|     city|order_id|order_date|amount|
+-------+---------+---------+--------+----------+------+
|      1|    Alice|Bangalore|     101|2025-11-01|  2500|
|      2|      Bob|    Delhi|     102|2025-11-02|  1800|
|      3|  Charlie|  Chennai|    NULL|      NULL|  NULL|
|      4|     NULL|     NULL|     103|2025-11-03|  2200|
+-------+---------+---------+--------+----------+------+



# Left Semi Join Returns only rows from left DataFrame that have a match in the right DataFrame — but doesn’t include columns from the right.

In [ ]:
semi_df = customers.join(orders, on="cust_id", how="left_semi") # “Which customers have placed orders?”
semi_df.show()


+-------+---------+---------+
|cust_id|cust_name|     city|
+-------+---------+---------+
|      1|    Alice|Bangalore|
|      2|      Bob|    Delhi|
+-------+---------+---------+



# Left Anti Join Returns rows from the left DataFrame that don’t have any match in the right.

In [ ]:
anti_df = customers.join(orders, on="cust_id", how="left_anti") # Which customers have not placed any orders
anti_df.show()


+-------+---------+-------+
|cust_id|cust_name|   city|
+-------+---------+-------+
|      3|  Charlie|Chennai|
+-------+---------+-------+



# Cross Join (Cartesian Product)  Every row of left × every row of right.

In [ ]:
cross_df = customers.crossJoin(orders)
cross_df.show()


+-------+---------+---------+--------+-------+----------+------+
|cust_id|cust_name|     city|order_id|cust_id|order_date|amount|
+-------+---------+---------+--------+-------+----------+------+
|      1|    Alice|Bangalore|     101|      1|2025-11-01|  2500|
|      1|    Alice|Bangalore|     102|      2|2025-11-02|  1800|
|      1|    Alice|Bangalore|     103|      4|2025-11-03|  2200|
|      2|      Bob|    Delhi|     101|      1|2025-11-01|  2500|
|      3|  Charlie|  Chennai|     101|      1|2025-11-01|  2500|
|      2|      Bob|    Delhi|     102|      2|2025-11-02|  1800|
|      2|      Bob|    Delhi|     103|      4|2025-11-03|  2200|
|      3|  Charlie|  Chennai|     102|      2|2025-11-02|  1800|
|      3|  Charlie|  Chennai|     103|      4|2025-11-03|  2200|
+-------+---------+---------+--------+-------+----------+------+



# Join Conditions with Multiple Columns

In [ ]:
df = customers.join(
    orders,
    (customers.cust_id == orders.cust_id) & (orders.amount > 2000)
    )
df.show()


+-------+---------+---------+--------+-------+----------+------+
|cust_id|cust_name|     city|order_id|cust_id|order_date|amount|
+-------+---------+---------+--------+-------+----------+------+
|      1|    Alice|Bangalore|     101|      1|2025-11-01|  2500|
+-------+---------+---------+--------+-------+----------+------+



In [ ]:
df = customers.join(orders,(customers.cust_id == orders.cust_id))
df.show()

+-------+---------+---------+--------+-------+----------+------+
|cust_id|cust_name|     city|order_id|cust_id|order_date|amount|
+-------+---------+---------+--------+-------+----------+------+
|      1|    Alice|Bangalore|     101|      1|2025-11-01|  2500|
|      2|      Bob|    Delhi|     102|      2|2025-11-02|  1800|
+-------+---------+---------+--------+-------+----------+------+



When joining two DataFrames:

One is large

One is small

Instead of shuffling both tables across the cluster
Spark broadcasts the small table to all worker nodes

This avoids heavy shuffle and improves performance.



When to Use Broadcast Join?

Use it when:

✔ One DataFrame is small (dimension table)
✔ Other DataFrame is large (fact table)
✔ Star schema type join

Example:

Orders table → 10 million rows

Country table → 200 rows

In [ ]:
# broadcast join

from pyspark.sql.functions import broadcast

optimized_join = orders.join(broadcast(customers), "cust_id", "inner")

optimized_join.show()

# Verify Join Execution Plan

orders.join(customers, "cust_id", "inner").explain(True)


+-------+--------+----------+------+---------+---------+
|cust_id|order_id|order_date|amount|cust_name|     city|
+-------+--------+----------+------+---------+---------+
|      1|     101|2025-11-01|  2500|    Alice|Bangalore|
|      2|     102|2025-11-02|  1800|      Bob|    Delhi|
+-------+--------+----------+------+---------+---------+

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [cust_id])
:- LogicalRDD [order_id#196L, cust_id#197L, order_date#198, amount#199L], false
+- LogicalRDD [cust_id#193L, cust_name#194, city#195], false

== Analyzed Logical Plan ==
cust_id: bigint, order_id: bigint, order_date: string, amount: bigint, cust_name: string, city: string
Project [cust_id#197L, order_id#196L, order_date#198, amount#199L, cust_name#194, city#195]
+- Join Inner, (cust_id#197L = cust_id#193L)
   :- LogicalRDD [order_id#196L, cust_id#197L, order_date#198, amount#199L], false
   +- LogicalRDD [cust_id#193L, cust_name#194, city#195], false

== Optimized Logical Plan ==
Project [c

# NULL HANDLING

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("NullHandlingExample").getOrCreate()

data = [
    ("Alice", None, 3500),
    ("Bob", 30, None),
    ("Cathy", None, 4000),
    ("David", 35, 2000),
    (None, 40, 5000)
]
columns = ["name", "age", "salary"]

df = spark.createDataFrame(data, columns)
df.show()


+-----+----+------+
| name| age|salary|
+-----+----+------+
|Alice|NULL|  3500|
|  Bob|  30|  NULL|
|Cathy|NULL|  4000|
|David|  35|  2000|
| NULL|  40|  5000|
+-----+----+------+



In [ ]:
# Detecting Nulls

df.filter(F.col("age").isNull()).show()


+-----+----+------+
| name| age|salary|
+-----+----+------+
|Alice|NULL|  3500|
|Cathy|NULL|  4000|
+-----+----+------+



In [8]:
df.filter(F.col("salary").isNotNull()).show()


NameError: name 'F' is not defined

In [7]:
# Replacing Nulls  Using fillna() / na.fill()

df.fillna({"age": 0, "salary": 1000, "name": "Unknown"}).show()




+-----+---+------+
| name|age|salary|
+-----+---+------+
|Alice| 25|  3000|
|  Bob| 30|  4000|
|Cathy| 22|  3500|
|David| 35|  2000|
+-----+---+------+



In [ ]:
# Dropping Nulls  dropna()

df.dropna().show()


In [ ]:
# Drop rows where all columns are null

df.dropna(how="all").show()


In [ ]:
# Drop rows where specific columns are null

df.dropna(subset=["name", "salary"]).show()


In [ ]:
# Using coalesce()

df.withColumn("effective_name", F.coalesce("name", F.lit("Unknown"))).show()
